# 改訂単体法

In [1]:
import numpy as np

### 最大係数規則

In [13]:
class RevisedSimplex:
    def __init__(self, A, b, c):
        self.A = A
        self.b = b
        self.c = c

        self.m, self.n = A.shape

        self.B_idx = list(range(self.n - self.m, self.n))
        self.N_idx = list(range(self.n - self.m))

    def solve(self):
        iteration = 0
        while True:
            # --- Step 1 ---
            B = self.A[:, self.B_idx] #基底行列
            N = self.A[:, self.N_idx] #非基底行列

            B_inv = np.linalg.inv(B) #基底行列の逆行列
            x_B = B_inv @ self.b  #初期の実行可能基底解
            b_bar = x_B
            
            if np.any(b_bar < 0):   # 実行可能かどうかのチェック
                print("初期実行可能解を見つけられません")
                return None, None, iteration
            # --- Step 2 ---
            c_B = self.c[self.B_idx]
            c_N = self.c[self.N_idx]

            c_bar = c_N - N.T @ B_inv.T @ c_B   # 被約費用の計算

            # --- Step 3 ---
            if np.all(c_bar <= 0): #c_barが全て非正であれば最適解
                x = np.zeros(self.n)
                x[self.B_idx] = b_bar
                return x, self.c @ x, iteration #最適解と最適値と反復回数を返して終了

            # c_barの中で最大のものを選び，対応する非基底変数を基底に入れる（最大係数規則）
            k = np.argmax(c_bar)
            entering = self.N_idx[k]

            # --- Step 4 ---
            a_k = self.A[:, entering]
            a_bar = B_inv @ a_k #N_barの非基底変数に対応する列ベクトル

            if np.all(a_bar <= 0):
               print("非有界です。")
               return None, np.inf, iteration #非有界なので終了

            # --- Step 5 ---
            theta = []
            for i in range(self.m):
                if a_bar[i] > 0:
                    theta.append(b_bar[i] / a_bar[i])
                else:
                    theta.append(np.inf)

            i_star = np.argmin(theta)
            leaving = self.B_idx[i_star]

            # 基底更新
            self.B_idx[i_star] = entering
            self.N_idx[k] = leaving
            iteration += 1

            if iteration > 10000: 
                print("反復回数が上限に達しました。")
                return None, None, iteration

例1

In [4]:
A = np.array([[1, 1, 1, 0, 0],
              [1, 3, 0, 1, 0],
              [2, 1, 0, 0, 1]])
b = np.array([6, 12, 10])
c = np.array([1, 2, 0, 0, 0])

例2　非有界のとき

In [6]:
A = np.array([
    [1, -2, 1, 0],
    [-1, 1, 0, 1]])
b = np.array([4, 2])
c = np.array([2, 1, 0, 0])

例3　巡回に陥るとき

In [11]:
A = np.array([[0.5, -5.5, -2.5, 9, 1, 0, 0], 
              [0.5, -1.5, -0.5, 1, 0, 1, 0],
              [1, 0, 0, 0, 0, 0, 1]])
b = np.array([0, 0, 1])
c = np.array([10, -57, -9, -24, 0, 0, 0])

例4 "簡単な"初期実行可能解がないとき($x_1=x_2=0$としても実行可能解が得られないとき)

In [9]:
A= np.array([[1, 1, 1, 0, 0],
                [1, 3, 0, 1, 0],
                [-3, -2, 0, 0, 1]])
b = np.array([6, 12, -6])
c = np.array([1, 2, 0, 0, 0])

In [12]:
solver = RevisedSimplex(A, b, c)
x_opt, val, iteration = solver.solve()
print("x* =", x_opt)
print("optimal value =", val)
print("iteration =", iteration)

x* = [1. 0. 1. 0. 2. 0. 0.]
optimal value = 1.0
iteration = 5


### 最小添字規則

In [10]:
class RevisedSimplex:
    def __init__(self, A, b, c):
        self.A = A
        self.b = b
        self.c = c

        self.m, self.n = A.shape

        self.B_idx = list(range(self.n - self.m, self.n))
        self.N_idx = list(range(self.n - self.m))

    def solve(self):
        iteration = 0
        while True:
        # --- Step 1 ---
         B = self.A[:, self.B_idx]
         N = self.A[:, self.N_idx]

         B_inv = np.linalg.inv(B)
         b_bar = B_inv @ self.b
         # 実行可能かどうかのチェック
         if np.any(b_bar < 0):
                print("初期実行可能解を見つけられません")
                return None, None, iteration
         

        # --- Step 2 ---
         c_B = self.c[self.B_idx]
         c_N = self.c[self.N_idx]

         c_bar = c_N - N.T @ B_inv.T @ c_B

        # --- Step 3 ---
         candidates = [j for j in range(len(c_bar)) if c_bar[j] > 0]

         if len(candidates) == 0:
            x = np.zeros(self.n)
            x[self.B_idx] = b_bar
            return x, self.c @ x, iteration

        # 最小添字規則（entering）
         k = min(candidates)
         entering = self.N_idx[k]

        # --- Step 4 ---
         a_k = self.A[:, entering]
         a_bar = B_inv @ a_k

         if np.all(a_bar <= 1e-10):
            return None, None, iteration  # 非有界

        # --- Step 5 ---
         theta = []
         for i in range(self.m):
            if a_bar[i] > 1e-10:
                theta.append(b_bar[i] / a_bar[i])
            else:
                theta.append(np.inf)

         theta_min = min(theta)

        # 最小添字規則（leaving）
         candidates = [i for i in range(self.m) if abs(theta[i] - theta_min) < 1e-12]
         i_star = min(candidates, key=lambda i: self.B_idx[i])

         leaving = self.B_idx[i_star]

        # 基底更新
         self.B_idx[i_star] = entering
         self.N_idx[k] = leaving
         iteration += 1

         if iteration > 10000:
            print("反復回数が上限に達しました。")
            return None, None, iteration
        

In [52]:
A = np.array([[0.5, -5.5, -2.5, 9, 1, 0, 0], 
              [0.5, -1.5, -0.5, 1, 0, 1, 0],
              [1, 0, 0, 0, 0, 0, 1]])
b = np.array([0, 0, 1])
c = np.array([10, -57, -9, -24, 0, 0, 0])

In [ ]:
solver = RevisedSimplex(A, b, c)
x_opt, val, iteration = solver.solve()
print("x* =", x_opt)
print("optimal value =", val)
print("iteration =", iteration)

反復回数が上限に達しました。


### クレー・ミンティの問題

In [14]:
def klee_minty(n):
    A = np.zeros((n, 2*n), dtype=float)
    b = np.zeros(n, dtype=float)
    c = np.zeros(2*n, dtype=float)

    for i in range(n):

        for j in range(i):

            if j == i:
                A[i, j] = 1
            else:
                A[i, j] =2*  10**(i-j)

        A[i, i] = 1

        A[i, n+i] = 1

        b[i] = 100**i

    for i in range(n):
        c[i] = 10**(n-i-1)

    return A, b, c

In [15]:
for k in range(1, 11):
    A, b, c = klee_minty(k)
    solver = RevisedSimplex(A, b, c)
    x_opt, val, iteration = solver.solve()
    print(f"n = {k}:")
    print("iteration =", iteration)

n = 1:
iteration = 1
n = 2:
iteration = 3
n = 3:
iteration = 7
n = 4:
iteration = 15
n = 5:
iteration = 31
n = 6:
iteration = 63
n = 7:
iteration = 127
n = 8:
iteration = 255
n = 9:
iteration = 511
n = 10:
iteration = 1023
